In [18]:
import polars as pl
pl.Config.set_tbl_rows(700)
pl.Config.set_tbl_cols(700)

polars.config.Config

### original mlp cv trainer

In [13]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import gc
from dataclasses import dataclass, field
from typing import Iterable, Optional
from time import perf_counter as now

import cudf
import rmm
import torch
import cupy as cp
import rmm.mr as mr
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import torch.nn as nn
import torch.nn.functional as F

from rmm.allocators.cupy import rmm_cupy_allocator
from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from torch.utils.dlpack import from_dlpack as torch_from_dlpack
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.utils.loggers import CVResult, CVLogger, NoOpLogger
from src.utils.print_duration import print_duration
from src.utils.mem_info import free_ram_gib, free_vram_gib

torch.cuda.manual_seed(self.seed)

dev_mr = mr.CudaAsyncMemoryResource()
mr.set_current_device_resource(dev_mr)
rmm.reinitialize(
    managed_memory=False,
    initial_pool_size=None,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

cp.get_default_memory_pool().set_limit(4 * 1024**3)
self.pmp = cp.cuda.PinnedMemoryPool()
cp.cuda.set_pinned_memory_allocator(self.pmp.malloc)


def compute_feature_stats(
    paths: list[str],
    features: list[str],
    num_cols: list[str],
    fold_col: str = None,
    include_folds: Optional[Iterable[int]] = None,
    exclude_folds: Optional[Iterable[int]] = None
):
    lf = pl.scan_parquet(paths, low_memory=True)
    if fold_col:
        if include_folds is not None:
            lf = lf.filter(pl.col(fold_col).is_in(sorted(include_folds)))
        if exclude_folds is not None:
            lf = lf.filter(~pl.col(fold_col).is_in(sorted(exclude_folds)))

    exprs = []
    for c in num_cols:
        exprs += [pl.col(c).cast(pl.Float64).mean().alias(f"{c}_mean"),
                  pl.col(c).cast(pl.Float64).std(ddof=0).alias(f"{c}_std")]
    out = lf.select(exprs).collect(engine="streaming")
    mean = out.select(
        [f"{c}_mean" for c in num_cols]).to_numpy().ravel().astype(np.float32)
    print(mean)
    std = out.select(
        [f"{c}_std" for c in num_cols]).to_numpy().ravel().astype(np.float32)
    std[std < 1e-4] = 1.0
    print(std)
    return mean, std


@dataclass
class ParquetStream(IterableDataset):
    paths: list[str] | str | os.PathLike

    features: list[str]
    target: str
    num_idxs: Iterable[int]

    mean: np.ndarray
    std: np.ndarray

    fold_col: Optional[str] = None
    include_folds: Optional[Iterable[int]] = None
    exclude_folds: Optional[Iterable[int]] = None
    weight_col: Optional[str] = None

    batch_size: int = 1024
    rows_per_epoch: int | None = None
    predict_mode: bool = False
    seed: int = 42
    shuffle: bool = True

    _epoch: int = field(init=False, default=0, repr=False)

    def __post_init__(self):
        super().__init__()

        # 形式正規化
        self.paths = [
            str(p)
            for p in (
                self.paths
                if isinstance(self.paths, (list, tuple))
                else [self.paths]
            )
        ]
        self.num_idxs = list(self.num_idxs or [])
        self.include_folds = (
            None
            if self.include_folds is None
            else set(self.include_folds)
        )
        self.exclude_folds = (
            None
            if self.exclude_folds is None
            else set(self.exclude_folds)
        )
        self.predict_mode = bool(self.predict_mode)

        # --- スキーマ取得は ParquetFile から（dataset 不使用）---
        pf0 = pq.ParquetFile(self.paths[0])
        all_cols = pf0.schema_arrow.names

        # 入力列（重複除去）
        cols = list(self.features or [])
        if (
            (not self.predict_mode)
            and (self.target in all_cols)
           ):
            cols.append(self.target)
        if (
            (not self.predict_mode)
            and self.weight_col
            and (self.weight_col in all_cols)
        ):
            cols.append(self.weight_col)
        if (
            (not self.predict_mode)
            and self.fold_col
            and (self.fold_col in all_cols)
        ):
            cols.append(self.fold_col)

        self._columns = list(dict.fromkeys(cols))

        self._norm_idxs = cp.asarray(
            self.num_idxs, dtype=cp.int64
        )
        if not (len(self.mean) == len(self.std) == len(self._norm_idxs)):
            raise ValueError(
                f"mean/std/num_idxs length mismatch: "
                f"{len(self.mean)}, {len(self.std)}, {len(self._norm_idxs)}"
            )
        self._mean_cu = cp.asarray(
            self.mean,
            dtype=cp.float32
        )
        self._std_cu = cp.asarray(
            self.std,
            dtype=cp.float32
        )

    def set_epoch(self, epoch: int):
        self._epoch = int(epoch)

    def _sharded_paths(self):
        info = get_worker_info()
        if info is None:
            return self.paths
        return self.paths[info.id::info.num_workers]

    def __iter__(self):
        info = get_worker_info()
        worker_id = info.id if info is not None else 0
        emitted = 0

        for path in self._sharded_paths():
            pf = pq.ParquetFile(path)
            seed = self.seed + self._epoch + worker_id
            if self.shuffle:
                rg_order = cp.asnumpy(
                    cp.random.RandomState(seed).permutation(pf.num_row_groups)
                )
            else:
                rg_order = np.arange(pf.num_row_groups, dtype=np.int64)

            carry_X = carry_y = carry_w = None

            for rg in rg_order:
                gdf = cudf.read_parquet(
                    path,
                    columns=self._columns,
                    row_groups=[int(rg)]
                ).astype("float32")
                if len(gdf) == 0:
                    continue

                # fold フィルタ（GPU）
                if (not self.predict_mode) and self.fold_col and (self.fold_col in gdf.columns):
                    if self.include_folds is not None:
                        gdf = gdf[gdf[self.fold_col].isin(sorted(self.include_folds))]
                    if self.exclude_folds is not None:
                        gdf = gdf[~gdf[self.fold_col].isin(sorted(self.exclude_folds))]
                    if len(gdf) == 0:
                        continue

                # GPU 内シャッフル
                if self.shuffle:
                    perm = cp.random.RandomState(seed).permutation(len(gdf))
                    gdf = gdf.take(cudf.Series(perm))

                # CuPy へ（ゼロコピー）
                X_cu = gdf[self.features].astype("float32").to_cupy()
                y_cu = (
                    gdf[self.target].astype("float32").values
                    if not self.predict_mode else None
                )
                w_cu = (
                    gdf[self.weight_col].astype("float32").values
                    if (self.weight_col and self.weight_col in gdf.columns)
                    else None
                )

                # 標準化（GPU, in-place）
                if self._norm_idxs.size > 0:
                    ni = self._norm_idxs
                    X_cu[:, ni] -= self._mean_cu
                    X_cu[:, ni] /= (self._std_cu + 1e-8)

                # 端数 carry を前段に連結（必要最小限）
                if carry_X is not None:
                    X_cu = cp.concatenate([carry_X, X_cu], axis=0)
                    if y_cu is not None:
                        y_cu = cp.concatenate([carry_y, y_cu], axis=0)
                    if w_cu is not None:
                        w_cu = cp.concatenate([carry_w, w_cu], axis=0)
                    carry_X = carry_y = carry_w = None

                m = X_cu.shape[0]
                full = (m // self.batch_size) * self.batch_size

                # バッチ生成
                for i in range(0, full, self.batch_size):
                    xb = X_cu[i:i+self.batch_size]
                    if self.predict_mode:
                        yield torch_from_dlpack(xb)
                    else:
                        yb = y_cu[i:i+self.batch_size]
                        if w_cu is not None:
                            wb = w_cu[i:i+self.batch_size]
                            yield (torch_from_dlpack(xb),
                                   torch_from_dlpack(yb).float(),
                                   torch_from_dlpack(wb).float())
                        else:
                            yield (torch_from_dlpack(xb),
                                   torch_from_dlpack(yb).float())
                    emitted += xb.shape[0]
                    if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                        return

                # 端数 carry
                rem = m - full
                if rem:
                    carry_X = X_cu[full:]
                    carry_y = y_cu[full:] if y_cu is not None else None
                    carry_w = w_cu[full:] if w_cu is not None else None

                # 後始末
                del gdf, X_cu
                if y_cu is not None:
                    del y_cu
                if w_cu is not None:
                    del w_cu
                gc.collect()

            # 最後の端数
            if carry_X is not None:
                if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                    return
                if self.predict_mode:
                    yield torch_from_dlpack(carry_X)
                else:
                    if carry_w is not None:
                        yield (torch_from_dlpack(carry_X),
                               torch_from_dlpack(carry_y).float(),
                               torch_from_dlpack(carry_w).float())
                    else:
                        yield (torch_from_dlpack(carry_X),
                               torch_from_dlpack(carry_y).float())


class SimpleMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout_rate,
        activation,
        num_idxs,
        cat_idxs,
        cat_dims
    ):
        super().__init__()
        self.num_idxs = num_idxs
        self.cat_idxs = cat_idxs

        self.embedding_layers = nn.ModuleList([
            nn.Embedding(
                num_embeddings=n, embedding_dim=min(50, n))
            for n in cat_dims
        ])

        total_embedding_dim = sum(
            min(50, n) for n in cat_dims
        )
        net_input_dim = len(num_idxs) + total_embedding_dim

        layers = []
        prev_dim = net_input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, xb):
        emb_list = [
            self.embedding_layers[i](xb[:, cat_idx].long())
            for i, cat_idx in enumerate(self.cat_idxs)
        ]
        x_emb = torch.cat(emb_list, dim=1) if emb_list else None

        # 数値部分
        x_num = xb[:, self.num_idxs]

        # 結合
        if x_emb is not None:
            x = torch.cat([x_num, x_emb], dim=1)
        else:
            x = x_num

        return self.net(x).squeeze(-1)


@dataclass
class MLPCVTrainer:
    data_id: int
    train_paths: str | list[str]
    test_paths: str | list[str] | None = None

    features: Optional[list[str]] = None

    target: str = "target"
    fold_col: Optional[str] = None
    weight_col: Optional[str] = None
    cat_cols: Optional[list[str]] = None

    params: dict = field(default_factory=dict)

    n_fold: int = 5
    seed: int = 42
    gpu: bool = True

    opts: dict = field(init=True, default_factory=dict)

    def __post_init__(self):
        if isinstance(self.train_paths, (str, os.PathLike)):
            self.train_paths = [str(self.train_paths)]
        else:
            self.train_paths = [str(p) for p in self.train_paths]

        if self.test_paths:
            if isinstance(self.test_paths, (str, os.PathLike)):
                self.test_paths = [str(self.test_paths)]
            else:
                self.test_paths = [str(p) for p in self.test_paths]

        default_params = {
            "lr": 1e-3,
            "batch_size": 256,
            "dropout_rate": 0.2,
            "hidden_dim1": 128,
            "hidden_dim2": 64,
            "hidden_dim3": None,
            "hidden_dim4": None,
            "max_epochs": 100,
            "min_epochs": 20,
            "activation": "ReLU",
            "early_stopping_rounds": 10,
            "t_max": 50,
            "eta_min": 1e-6,
            "device": "cuda"
        }

        ACTIVATION_MAPPING = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "GELU": nn.GELU,
            "SiLU": nn.SiLU,
            "Tanh": nn.Tanh,
            "Sigmoid": nn.Sigmoid,
        }

        self.params = {**default_params, **self.params}

        self.params["activation"] = ACTIVATION_MAPPING[self.params["activation"]]

        hidden_dims = []
        i = 1
        while f"hidden_dim{i}" in self.params:
            dim = self.params[f"hidden_dim{i}"]
            if dim is None or dim == -1:
                break
            hidden_dims.append(dim)
            i += 1

        self.params["hidden_dims"] = hidden_dims

        hdr = pl.read_parquet(self.train_paths, n_rows=0)
        all_cols = hdr.columns

        if self.fold_col is None:
            self.fold_col = f"{self.n_fold}fold-s{self.seed}"

        if self.cat_cols is None:
            self.cat_cols = [
                c for c, dt in zip(hdr.columns, hdr.dtypes)
                if dt == pl.Categorical
            ]

        if self.fold_col not in all_cols:
            raise ValueError(f"fold_col not found in dataset: {self.fold_col}")
        else:
            print(f"Fold Col: {self.fold_col}")

        if self.features is None:
            meta = {"row_id"}
            if self.target in all_cols:
                meta.add(self.target)
            if self.weight_col in all_cols:
                meta.add(self.weight_col)
            if self.fold_col:
                meta.add(self.fold_col)

            self.features = [
                c for c in all_cols
                if c not in list(meta) + ['day_max_by_job', 'pdays_median_by_job', 'previous_median_by_job', 'day_max_by_marital', 'campaign_median_by_marital', 'pdays_median_by_marital', 'previous_median_by_marital', 'age_max_by_education', 'day_max_by_education', 'campaign_median_by_education', 'pdays_median_by_education', 'previous_median_by_education', 'age_median_by_default', 'day_max_by_default', 'campaign_median_by_default', 'pdays_median_by_default', 'previous_median_by_default', 'day_max_by_housing', 'campaign_median_by_housing', 'pdays_max_by_housing', 'pdays_median_by_housing', 'previous_median_by_housing', 'age_median_by_loan', 'day_max_by_loan', 'day_median_by_loan', 'campaign_median_by_loan', 'pdays_median_by_loan', 'previous_median_by_loan', 'day_max_by_contact', 'campaign_median_by_contact', 'pdays_max_by_contact', 'pdays_median_by_contact', 'previous_median_by_contact', 'day_max_by_month', 'pdays_median_by_month', 'previous_median_by_month', 'day_max_by_poutcome', 'balance_std_by_age2', 'day_std_by_age2', 'duration_std_by_age2', 'campaign_std_by_age2', 'pdays_std_by_age2', 'previous_std_by_age2', 'age_std_by_balance2', 'day_std_by_balance2', 'duration_std_by_balance2', 'campaign_std_by_balance2', 'pdays_std_by_balance2', 'previous_std_by_balance2', 'pdays_median_by_day2', 'previous_median_by_day2', 'age_std_by_duration2', 'balance_std_by_duration2', 'day_std_by_duration2', 'campaign_std_by_duration2', 'pdays_std_by_duration2', 'previous_std_by_duration2', 'age_std_by_campaign2', 'balance_std_by_campaign2', 'day_std_by_campaign2', 'duration_std_by_campaign2', 'pdays_std_by_campaign2', 'pdays_median_by_campaign2', 'previous_std_by_campaign2', 'previous_median_by_campaign2', 'age_std_by_pdays2', 'balance_std_by_pdays2', 'duration_std_by_pdays2', 'campaign_std_by_pdays2', 'previous_std_by_pdays2', 'age_std_by_previous2', 'balance_std_by_previous2', 'day_std_by_previous2', 'duration_std_by_previous2', 'campaign_std_by_previous2', 'pdays_std_by_previous2'] and "fold" not in c
            ]

        self.num_cols = [
            col for col in self.features
            if col not in self.cat_cols
        ]

        self.cat_idxs = [self.features.index(c) for c in self.cat_cols]
        self.num_idxs = [self.features.index(c) for c in self.num_cols]

        print("cat_idxs", self.cat_idxs)
        scan = pl.scan_parquet(self.train_paths)
        exprs = [
            pl.col(c)
            .rank("dense")
            .cast(pl.Int32)
            .n_unique()
            .alias(c) for c in self.cat_cols
        ]
        df1 = scan.select(exprs).collect()

        if df1.width == 0 or df1.height == 0:
            self.cat_dims = []
        else:
            self.cat_dims = [int(x) if x is not None else 0 for x in df1.row(0)]

        print("cat_dims", self.cat_dims)
        torch.cuda.manual_seed(self.seed)

        dev_mr = mr.CudaAsyncMemoryResource()
        mr.set_current_device_resource(dev_mr)
        rmm.reinitialize(
            managed_memory=False,
            initial_pool_size=None,
        )
        cp.cuda.set_allocator(rmm_cupy_allocator)

        cp.get_default_memory_pool().set_limit(4 * 1024**3)
        self.pmp = cp.cuda.PinnedMemoryPool()
        cp.cuda.set_pinned_memory_allocator(self.pmp.malloc)

    def fit_one_fold(
        self,
        fold_idx=0,
        loggers=None
    ):
        print(f"Free CPU Mem: {round(free_ram_gib(), 2)} GB")
        print(f"Free GPU Mem: {round(free_vram_gib(), 2)} GB")

        mean, std = compute_feature_stats(
            self.train_paths,
            self.features,
            self.num_cols,
            self.fold_col,
            exclude_folds=[fold_idx]
        )

        train_ds = ParquetStream(
            self.train_paths,
            self.features,
            self.target,
            self.num_idxs,
            mean,
            std,
            fold_col=self.fold_col,
            exclude_folds=[fold_idx],
            weight_col=self.weight_col,
            batch_size=self.params["batch_size"],
            predict_mode=False,
            seed=self.seed,
            shuffle=True
        )
        valid_ds = ParquetStream(
            self.train_paths,
            self.features,
            self.target,
            self.num_idxs,
            mean,
            std,
            fold_col=self.fold_col,
            include_folds=[fold_idx],
            batch_size=self.params["batch_size"],
            predict_mode=False,
            seed=self.seed,
            shuffle=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=None,
            num_workers=0,
            shuffle=False
        )
        val_loader = DataLoader(
            valid_ds,
            batch_size=None,
            num_workers=0,
            shuffle=False
        )

        lf = pl.scan_parquet(self.train_paths)
        val_y = (
            lf.filter(pl.col(self.fold_col) == fold_idx)
            .select(self.target)
            .collect(engine="streaming")
            .to_series()
            .to_numpy()
            .astype("float32")
        )

        model = SimpleMLP(
            input_dim=len(self.features),
            hidden_dims=self.params["hidden_dims"],
            dropout_rate=self.params["dropout_rate"],
            activation=self.params["activation"],
            num_idxs=self.num_idxs,
            cat_idxs=self.cat_idxs,
            cat_dims=self.cat_dims
        ).to(self.params["device"])

        optimizer = torch.optim.Adam(model.parameters(), lr=self.params["lr"])
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=self.params["t_max"],
            eta_min=self.params["eta_min"]
        )

        best_log_loss = float("inf")
        best_model_state = None
        best_epoch = 0

        for epoch in range(3):
            model.train()
            for batch in train_loader:
                if len(batch) == 3:
                    xb, yb, wb = batch
                else:
                    xb, yb = batch
                    wb = None

                preds = model(xb)

                if wb is None:
                    loss = F.binary_cross_entropy_with_logits(
                        preds, yb, reduction="mean"
                    )
                else:
                    loss = F.binary_cross_entropy_with_logits(
                        preds, yb, weight=wb, reduction="mean"
                    )
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            preds = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    pred_logits = model(xb)
                    pred_probs = torch.sigmoid(pred_logits).cpu().numpy()
                    preds.append(pred_probs)
            val_pred = np.concatenate(preds)
            val_log_loss = log_loss(val_y, val_pred)
            scheduler.step()

            train_pred = []
            train_y = []
            with torch.no_grad():
                for batch in train_loader:
                    if len(batch) == 3:
                        xb, yb, wb = batch
                    else:
                        xb, yb = batch
                        wb = None
                    xb = xb.to(self.params["device"])
                    pred_logits = model(xb)
                    pred_probs = torch.sigmoid(
                        pred_logits).cpu().numpy()
                    train_pred.append(pred_probs)
                    train_y.append(yb.cpu().numpy())
            train_pred = np.concatenate(train_pred)
            train_y = np.concatenate(train_y)

            train_log_loss = log_loss(train_y, train_pred)

            print(
                f"Epoch {epoch+1}: "
                f"Train Logloss = {train_log_loss:.5f}, "
                f"Val Logloss = {val_log_loss:.5f}"
            )

            if val_log_loss < best_log_loss:
                best_log_loss = val_log_loss
                best_model_state = {
                    k: v.cpu().clone() for k, v
                    in model.state_dict().items()
                }
                best_epoch = epoch + 1
                print(
                    f"New best model saved at epoch {epoch+1}, "
                    f"Logloss: {val_log_loss:.5f}")
            elif (
                (epoch - best_epoch >= self.params["early_stopping_rounds"])
                and (epoch + 1 >= self.params["min_epochs"])
            ):
                print(f"Early stopping at epoch {epoch+1}")
                print(f"Loading best model from epoch {best_epoch} "
                      f"with Logloss {best_log_loss:.5f}")
                break

        model.load_state_dict(
            {
                k: v.to(self.params["device"])
                for k, v in best_model_state.items()
            }
        )

        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(self.params["device"])
                val_logits = model(xb)
                val_probs = torch.sigmoid(val_logits).cpu().numpy()
                val_preds.append(val_probs)

        val_preds = np.concatenate(val_preds).ravel()

        best_auc = roc_auc_score(val_y, val_preds)

        print(f"Best Logloss: {best_log_loss:.5f}")
        print(f"Best AUC: {best_auc: .5f}")

        print(f"Free CPU Mem: {round(free_ram_gib(), 2)} GB")
        print(f"Free GPU Mem: {round(free_vram_gib(), 2)} GB")

        del model
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
        self.pmp.free_all_blocks()

### debug code

In [14]:
import gc
import os
from dataclasses import dataclass, field
from typing import Iterable, Optional
from time import perf_counter as now

import cudf
import rmm
import torch
import cupy as cp
import rmm.mr as mr
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import torch.nn as nn
import torch.nn.functional as F

from rmm.allocators.cupy import rmm_cupy_allocator
from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from torch.utils.dlpack import from_dlpack as torch_from_dlpack
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.utils.loggers import CVResult, CVLogger, NoOpLogger
from src.utils.print_duration import print_duration
from src.utils.mem_info import free_ram_gib, free_vram_gib


def compute_feature_stats(
    paths: list[str],
    features: list[str],
    num_cols: list[str],
    fold_col: str = None,
    include_folds: Optional[Iterable[int]] = None,
    exclude_folds: Optional[Iterable[int]] = None
):
    lf = pl.scan_parquet(paths, low_memory=True)
    if fold_col:
        if include_folds is not None:
            lf = lf.filter(pl.col(fold_col).is_in(sorted(include_folds)))
        if exclude_folds is not None:
            lf = lf.filter(~pl.col(fold_col).is_in(sorted(exclude_folds)))

    exprs = []
    for c in num_cols:
        exprs += [pl.col(c).cast(pl.Float64).mean().alias(f"{c}_mean"),
                  pl.col(c).cast(pl.Float64).std(ddof=0).alias(f"{c}_std")]
    out = lf.select(exprs).collect(engine="streaming")
    mean = out.select(
        [f"{c}_mean" for c in num_cols]).to_numpy().ravel().astype(np.float32)
    print(mean)
    std = out.select(
        [f"{c}_std" for c in num_cols]).to_numpy().ravel().astype(np.float32)
    std[std < 1e-4] = 1.0
    std = np.nan_to_num(std, nan=1.0)
    print(std)
    return mean, std


@dataclass
class ParquetStream(IterableDataset):
    paths: list[str] | str | os.PathLike

    features: list[str]
    target: str
    num_idxs: Iterable[int]

    mean: np.ndarray
    std: np.ndarray

    fold_col: Optional[str] = None
    include_folds: Optional[Iterable[int]] = None
    exclude_folds: Optional[Iterable[int]] = None
    weight_col: Optional[str] = None

    batch_size: int = 1024
    rows_per_epoch: int | None = None
    predict_mode: bool = False
    seed: int = 42
    shuffle: bool = True

    _epoch: int = field(init=False, default=0, repr=False)

    def __post_init__(self):
        super().__init__()

        # 形式正規化
        self.paths = [
            str(p)
            for p in (
                self.paths
                if isinstance(self.paths, (list, tuple))
                else [self.paths]
            )
        ]
        self.num_idxs = list(self.num_idxs or [])
        self.include_folds = (
            None
            if self.include_folds is None
            else set(self.include_folds)
        )
        self.exclude_folds = (
            None
            if self.exclude_folds is None
            else set(self.exclude_folds)
        )
        self.predict_mode = bool(self.predict_mode)

        # --- スキーマ取得は ParquetFile から（dataset 不使用）---
        pf0 = pq.ParquetFile(self.paths[0])
        all_cols = pf0.schema_arrow.names

        # 入力列（重複除去）
        cols = list(self.features or [])
        if (
            (not self.predict_mode)
            and (self.target in all_cols)
           ):
            cols.append(self.target)
        if (
            (not self.predict_mode)
            and self.weight_col
            and (self.weight_col in all_cols)
        ):
            cols.append(self.weight_col)
        if (
            (not self.predict_mode)
            and self.fold_col
            and (self.fold_col in all_cols)
        ):
            cols.append(self.fold_col)

        self._columns = list(dict.fromkeys(cols))

        self._norm_idxs = cp.asarray(
            self.num_idxs, dtype=cp.int64
        )
        if not (len(self.mean) == len(self.std) == len(self._norm_idxs)):
            raise ValueError(
                f"mean/std/num_idxs length mismatch: "
                f"{len(self.mean)}, {len(self.std)}, {len(self._norm_idxs)}"
            )
        self._mean_cu = cp.asarray(
            self.mean,
            dtype=cp.float32
        )
        self._std_cu = cp.asarray(
            self.std,
            dtype=cp.float32
        )

    def set_epoch(self, epoch: int):
        self._epoch = int(epoch)

    def _sharded_paths(self):
        info = get_worker_info()
        if info is None:
            return self.paths
        return self.paths[info.id::info.num_workers]

    def __iter__(self):
        info = get_worker_info()
        worker_id = info.id if info is not None else 0
        emitted = 0

        for path in self._sharded_paths():
            pf = pq.ParquetFile(path)
            seed = self.seed + self._epoch + worker_id
            if self.shuffle:
                rg_order = cp.asnumpy(
                    cp.random.RandomState(seed).permutation(pf.num_row_groups)
                )
            else:
                rg_order = np.arange(pf.num_row_groups, dtype=np.int64)

            carry_X = carry_y = carry_w = None

            for rg in rg_order:
                gdf = cudf.read_parquet(
                    path,
                    columns=self._columns,
                    row_groups=[int(rg)]
                ).astype("float32")
                if len(gdf) == 0:
                    continue

                # fold フィルタ（GPU）
                if (not self.predict_mode) and self.fold_col and (self.fold_col in gdf.columns):
                    if self.include_folds is not None:
                        gdf = gdf[gdf[self.fold_col].isin(sorted(self.include_folds))]
                    if self.exclude_folds is not None:
                        gdf = gdf[~gdf[self.fold_col].isin(sorted(self.exclude_folds))]
                    if len(gdf) == 0:
                        continue

                # GPU 内シャッフル
                if self.shuffle:
                    perm = cp.random.RandomState(seed).permutation(len(gdf))
                    gdf = gdf.take(cudf.Series(perm))

                # CuPy へ（ゼロコピー）
                X_cu = gdf[self.features].astype("float32").to_cupy()
                y_cu = (
                    gdf[self.target].astype("float32").values
                    if not self.predict_mode else None
                )
                w_cu = (
                    gdf[self.weight_col].astype("float32").values
                    if (self.weight_col and self.weight_col in gdf.columns)
                    else None
                )

                # 標準化（GPU, in-place）
                if self._norm_idxs.size > 0:
                    ni = self._norm_idxs
                    X_cu[:, ni] -= self._mean_cu
                    X_cu[:, ni] /= (self._std_cu + 1e-8)

                # 端数 carry を前段に連結（必要最小限）
                if carry_X is not None:
                    X_cu = cp.concatenate([carry_X, X_cu], axis=0)
                    if y_cu is not None:
                        y_cu = cp.concatenate([carry_y, y_cu], axis=0)
                    if w_cu is not None:
                        w_cu = cp.concatenate([carry_w, w_cu], axis=0)
                    carry_X = carry_y = carry_w = None

                m = X_cu.shape[0]
                full = (m // self.batch_size) * self.batch_size

                # バッチ生成
                for i in range(0, full, self.batch_size):
                    xb = X_cu[i:i+self.batch_size]
                    if self.predict_mode:
                        yield torch_from_dlpack(xb)
                    else:
                        yb = y_cu[i:i+self.batch_size]
                        if w_cu is not None:
                            wb = w_cu[i:i+self.batch_size]
                            yield (torch_from_dlpack(xb),
                                   torch_from_dlpack(yb).float(),
                                   torch_from_dlpack(wb).float())
                        else:
                            yield (torch_from_dlpack(xb),
                                   torch_from_dlpack(yb).float())
                    emitted += xb.shape[0]
                    if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                        return

                # 端数 carry
                rem = m - full
                if rem:
                    carry_X = X_cu[full:]
                    carry_y = y_cu[full:] if y_cu is not None else None
                    carry_w = w_cu[full:] if w_cu is not None else None

                # 後始末
                del gdf, X_cu
                if y_cu is not None:
                    del y_cu
                if w_cu is not None:
                    del w_cu
                gc.collect()

            # 最後の端数
            if carry_X is not None:
                if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                    return
                if self.predict_mode:
                    yield torch_from_dlpack(carry_X)
                else:
                    if carry_w is not None:
                        yield (torch_from_dlpack(carry_X),
                               torch_from_dlpack(carry_y).float(),
                               torch_from_dlpack(carry_w).float())
                    else:
                        yield (torch_from_dlpack(carry_X),
                               torch_from_dlpack(carry_y).float())


class SimpleMLP(nn.Module):
    """
    Parameters
    ----------
    input_dim : int
        数値特徴量の次元数（カテゴリ変数を除いたもの）
    hidden_dims : list of int
        MLPの隠れ層サイズリスト
    dropout_rate : float
        ドロップアウト率
    activation : nn.Module
        活性化関数
    num_idxs : list
        数値変数のインデックスのリスト
    cat_idxs : list
        カテゴリ変数のインデックスのリスト
    cat_dims : list
        カテゴリ変数のユニークな値のリスト
    """

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout_rate,
        activation,
        num_idxs,
        cat_idxs,
        cat_dims
    ):
        super().__init__()
        self.num_idxs = num_idxs
        self.cat_idxs = cat_idxs

        self.embedding_layers = nn.ModuleList([
            nn.Embedding(
                num_embeddings=n, embedding_dim=min(50, (n + 1) // 2))
            for n in cat_dims
        ])

        total_embedding_dim = sum(
            min(50, (n + 1) // 2) for n in cat_dims
        )
        net_input_dim = len(num_idxs) + total_embedding_dim

        layers = []
        prev_dim = net_input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, xb):
        """
        Parameters
        ----------
        xb : torch.Tensor
            数値特徴量とカテゴリ特徴量を統合したもの

        Returns
        -------
        torch.Tensor
            (B,) の予測
        """
        emb_list = [
            self.embedding_layers[i](xb[:, cat_idx].long())
            for i, cat_idx in enumerate(self.cat_idxs)
        ]
        x_emb = torch.cat(emb_list, dim=1) if emb_list else None

        # 数値部分
        x_num = xb[:, self.num_idxs]

        # 結合
        if x_emb is not None:
            x = torch.cat([x_num, x_emb], dim=1)
        else:
            x = x_num

        return self.net(x).squeeze(-1)


@dataclass
class MLPCVTrainer:
    data_id: int
    train_paths: str | list[str]
    test_paths: str | list[str] | None = None

    features: Optional[list[str]] = None

    target: str = "target"
    fold_col: Optional[str] = None
    weight_col: Optional[str] = None
    cat_cols: Optional[list[str]] = None

    params: dict = field(default_factory=dict)

    n_fold: int = 5
    seed: int = 42
    gpu: bool = True

    opts: dict = field(init=True, default_factory=dict)

    def __post_init__(self):
        if isinstance(self.train_paths, (str, os.PathLike)):
            self.train_paths = [str(self.train_paths)]
        else:
            self.train_paths = [str(p) for p in self.train_paths]

        if self.test_paths:
            if isinstance(self.test_paths, (str, os.PathLike)):
                self.test_paths = [str(self.test_paths)]
            else:
                self.test_paths = [str(p) for p in self.test_paths]

        default_params = {
            "lr": 1e-3,
            "batch_size": 256,
            "dropout_rate": 0.2,
            "hidden_dim1": 128,
            "hidden_dim2": 64,
            "hidden_dim3": None,
            "hidden_dim4": None,
            "max_epochs": 100,
            "min_epochs": 20,
            "activation": "ReLU",
            "early_stopping_rounds": 10,
            "t_max": 50,
            "eta_min": 1e-6,
            "device": "cuda"
        }

        ACTIVATION_MAPPING = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "GELU": nn.GELU,
            "SiLU": nn.SiLU,
            "Tanh": nn.Tanh,
            "Sigmoid": nn.Sigmoid,
        }

        self.params = {**default_params, **self.params}

        self.params["activation"] = ACTIVATION_MAPPING[self.params["activation"]]

        hidden_dims = []
        i = 1
        while f"hidden_dim{i}" in self.params:
            dim = self.params[f"hidden_dim{i}"]
            if dim is None or dim == -1:
                break
            hidden_dims.append(dim)
            i += 1

        self.params["hidden_dims"] = hidden_dims

        hdr = pl.read_parquet(self.train_paths, n_rows=0)
        all_cols = hdr.columns

        if self.fold_col is None:
            self.fold_col = f"{self.n_fold}fold-s{self.seed}"

        if self.cat_cols is None:
            self.cat_cols = [
                c for c, dt in zip(hdr.columns, hdr.dtypes)
                if dt == pl.Categorical
            ]

        if self.fold_col not in all_cols:
            raise ValueError(f"fold_col not found in dataset: {self.fold_col}")
        else:
            print(f"Fold Col: {self.fold_col}")

        if self.features is None:
            meta = {"row_id"}
            if self.target in all_cols:
                meta.add(self.target)
            if self.weight_col in all_cols:
                meta.add(self.weight_col)
            if self.fold_col:
                meta.add(self.fold_col)

            self.features = [
                c for c in all_cols
                if c not in meta and "fold" not in c
            ]

        self.num_cols = [
            col for col in self.features
            if col not in self.cat_cols
        ]

        self.cat_idxs = [self.features.index(c) for c in self.cat_cols]
        self.num_idxs = [self.features.index(c) for c in self.num_cols]

        scan = pl.scan_parquet(self.train_paths)
        exprs = [
            pl.col(c)
            .rank("dense")
            .cast(pl.Int32)
            .n_unique()
            .alias(c) for c in self.cat_cols
        ]
        df1 = scan.select(exprs).collect()

        if df1.width == 0 or df1.height == 0:
            self.cat_dims = []
        else:
            self.cat_dims = [int(x) if x is not None else 0 for x in df1.row(0)]

    def fit_one_fold(
        self,
        fold_idx=0,
        loggers=None
    ):
        """
        指定した1つのfoldのみを用いてモデルを学習する。
        主にOptunaによるハイパーパラメータ探索時に使用。

        Parameters
        ----------
        fold : int
            学習に使うfold番号。

        Rerurn
        ------
        best_logloss : float
            Score
        """
        t_total_start = now()

        loggers = loggers or [NoOpLogger()]
        meta = {
            "data_id": self.data_id,
            "seed": self.seed,
            "n_fold": self.n_fold,
            **self.params
        }
        for lg in loggers:
            lg.on_start(meta)

        print(f"Free CPU Mem: {round(free_ram_gib(), 2)} GB")
        print(f"Free GPU Mem: {round(free_vram_gib(), 2)} GB")

        mean, std = compute_feature_stats(
            self.train_paths,
            self.features,
            self.num_cols,
            self.fold_col,
            exclude_folds=[fold_idx]
        )

        train_ds = ParquetStream(
            self.train_paths,
            self.features,
            self.target,
            self.num_idxs,
            mean,
            std,
            fold_col=self.fold_col,
            exclude_folds=[fold_idx],
            weight_col=self.weight_col,
            batch_size=self.params["batch_size"],
            predict_mode=False,
            seed=self.seed,
            shuffle=True
        )
        valid_ds = ParquetStream(
            self.train_paths,
            self.features,
            self.target,
            self.num_idxs,
            mean,
            std,
            fold_col=self.fold_col,
            include_folds=[fold_idx],
            batch_size=self.params["batch_size"],
            predict_mode=False,
            seed=self.seed,
            shuffle=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=None,
            num_workers=0,
            shuffle=False
        )
        val_loader = DataLoader(
            valid_ds,
            batch_size=None,
            num_workers=0,
            shuffle=False
        )

        lf = pl.scan_parquet(self.train_paths)
        val_y = (
            lf.filter(pl.col(self.fold_col) == fold_idx)
            .select(self.target)
            .collect(engine="streaming")
            .to_series()
            .to_numpy()
            .astype("float32")
        )

        model = SimpleMLP(
            input_dim=len(self.features),
            hidden_dims=self.params["hidden_dims"],
            dropout_rate=self.params["dropout_rate"],
            activation=self.params["activation"],
            num_idxs=self.num_idxs,
            cat_idxs=self.cat_idxs,
            cat_dims=self.cat_dims
        ).to(self.params["device"])

        optimizer = torch.optim.Adam(model.parameters(), lr=self.params["lr"])
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=self.params["t_max"],
            eta_min=self.params["eta_min"]
        )

        best_log_loss = float("inf")
        best_model_state = None
        best_epoch = 0

        history = {
            "train": {"loss": [], "auc": []},
            "valid": {"loss": [], "auc": []}
        }
        extra_hist = {"lr": []}

        for epoch in range(self.params["max_epochs"]):
            model.train()
            for batch in train_loader:
                if len(batch) == 3:
                    xb, yb, wb = batch
                else:
                    xb, yb = batch
                    wb = None

                preds = model(xb)

                if wb is None:
                    loss = F.binary_cross_entropy_with_logits(
                        preds, yb, reduction="mean"
                    )
                else:
                    loss = F.binary_cross_entropy_with_logits(
                        preds, yb, weight=wb, reduction="mean"
                    )
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            preds = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    pred_logits = model(xb)
                    pred_probs = torch.sigmoid(pred_logits).cpu().numpy()
                    preds.append(pred_probs)
            val_pred = np.concatenate(preds)
            val_log_loss = log_loss(val_y, val_pred)
            scheduler.step()
            lr = scheduler.get_last_lr()[0]

            train_pred = []
            train_y = []
            with torch.no_grad():
                for batch in train_loader:
                    if len(batch) == 3:
                        xb, yb, wb = batch
                    else:
                        xb, yb = batch
                        wb = None
                    xb = xb.to(self.params["device"])
                    pred_logits = model(xb)
                    pred_probs = torch.sigmoid(
                        pred_logits).cpu().numpy()
                    train_pred.append(pred_probs)
                    train_y.append(yb.cpu().numpy())
            train_pred = np.concatenate(train_pred)
            train_y = np.concatenate(train_y)

            train_log_loss = log_loss(train_y, train_pred)
            train_auc = roc_auc_score(train_y, train_pred)

            history["train"]["loss"].append(train_log_loss)
            history["train"]["auc"].append(train_auc)

            val_auc = roc_auc_score(val_y, val_pred)
            history["valid"]["loss"].append(val_log_loss)
            history["valid"]["auc"].append(val_auc)

            extra_hist["lr"].append(lr)

            print(
                f"Epoch {epoch+1}: "
                f"Train Logloss = {train_log_loss:.5f}, "
                f"Val Logloss = {val_log_loss:.5f}"
            )

            if val_log_loss < best_log_loss:
                best_log_loss = val_log_loss
                best_model_state = {
                    k: v.cpu().clone() for k, v
                    in model.state_dict().items()
                }
                best_epoch = epoch + 1
                print(
                    f"New best model saved at epoch {epoch+1}, "
                    f"Logloss: {val_log_loss:.5f}")
            elif (
                (epoch - best_epoch >= self.params["early_stopping_rounds"])
                and (epoch + 1 >= self.params["min_epochs"])
            ):
                print(f"Early stopping at epoch {epoch+1}")
                print(f"Loading best model from epoch {best_epoch} "
                      f"with Logloss {best_log_loss:.5f}")
                break

        model.load_state_dict(
            {
                k: v.to(self.params["device"])
                for k, v in best_model_state.items()
            }
        )

        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(self.params["device"])
                val_logits = model(xb)
                val_probs = torch.sigmoid(val_logits).cpu().numpy()
                val_preds.append(val_probs)

        val_preds = np.concatenate(val_preds).ravel()

        best_auc = roc_auc_score(val_y, val_preds)

        print(f"Best Logloss: {best_log_loss:.5f}")
        print(f"Best AUC: {best_auc: .5f}")

        t_total_end = now()
        runtime = print_duration(
            t_total_start, t_total_end, "Total CV Runtime"
        )
        print(f"Free CPU Mem: {round(free_ram_gib(), 2)} GB")
        print(f"Free GPU Mem: {round(free_vram_gib(), 2)} GB")

        fold_summary = {
            "loss": best_log_loss,
            "auc": best_auc,
            "runtime": runtime
        }

        for lg in loggers:
            lg.on_fold_end(
                fold_idx,
                "epoch",
                history,
                extra_hist,
                fold_summary
            )
        del model
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
        self.pmp.free_all_blocks()

        return best_auc


### input cell

In [15]:
trainer = MLPCVTrainer(
    "test",
    train_paths="../../artifacts/features/040/train.parquet"
)
trainer.fit_one_fold()

Fold Col: 5fold-s42
Free CPU Mem: 13.78 GB
Free GPU Mem: 5.12 GB
[ 4.09185791e+01  1.20330750e+03  1.61166801e+01  2.56230957e+02
  2.57685328e+00  2.24294052e+01  2.98604995e-01  4.09258385e+01
  8.83687115e+00  8.59916153e+01  3.92336578e+01  1.20237866e+03
  2.72721362e+03  9.74178281e+04  6.40855042e+02  1.61169395e+01
  8.23340988e+00  3.10000000e+01  1.63433208e+01  2.56011353e+02
  2.71539581e+02  4.83294434e+03  1.35143250e+02  2.57588887e+00
  2.70417929e+00  5.59156685e+01  1.98413670e+00  2.23823318e+01
  7.66884232e+01  8.54758057e+02 -1.00000000e+00  2.99923241e-01
  1.34245777e+00  8.16279602e+01  0.00000000e+00  4.09237213e+01
  8.94025993e+00  9.42203293e+01  3.96977844e+01  1.20250940e+03
  2.80785889e+03  9.77564688e+04  6.32500122e+02  1.61168518e+01
  8.25034809e+00  3.10000000e+01  1.67401104e+01  2.56011963e+02
  2.71713226e+02  4.90472266e+03  1.32737350e+02  2.57613635e+00
  2.71536493e+00  5.90901604e+01  2.00000000e+00  2.23801003e+01
  7.70463562e+01  8.69318

ValueError: Input contains NaN.

In [19]:
df1 = pl.read_parquet("../../artifacts/features/040/train.parquet")

In [20]:
df1.describe()

statistic,row_id,age,balance,day,duration,campaign,pdays,previous,age_mean_by_job,age_std_by_job,age_max_by_job,age_median_by_job,balance_mean_by_job,balance_std_by_job,balance_max_by_job,balance_median_by_job,day_mean_by_job,day_std_by_job,day_max_by_job,day_median_by_job,duration_mean_by_job,duration_std_by_job,duration_max_by_job,duration_median_by_job,campaign_mean_by_job,campaign_std_by_job,campaign_max_by_job,campaign_median_by_job,pdays_mean_by_job,pdays_std_by_job,pdays_max_by_job,pdays_median_by_job,previous_mean_by_job,previous_std_by_job,previous_max_by_job,previous_median_by_job,age_mean_by_marital,age_std_by_marital,age_max_by_marital,age_median_by_marital,balance_mean_by_marital,balance_std_by_marital,balance_max_by_marital,balance_median_by_marital,day_mean_by_marital,day_std_by_marital,day_max_by_marital,day_median_by_marital,duration_mean_by_marital,duration_std_by_marital,duration_max_by_marital,duration_median_by_marital,campaign_mean_by_marital,campaign_std_by_marital,campaign_max_by_marital,campaign_median_by_marital,pdays_mean_by_marital,pdays_std_by_marital,pdays_max_by_marital,pdays_median_by_marital,previous_mean_by_marital,previous_std_by_marital,previous_max_by_marital,previous_median_by_marital,age_mean_by_education,age_std_by_education,age_max_by_education,age_median_by_education,balance_mean_by_education,balance_std_by_education,balance_max_by_education,balance_median_by_education,day_mean_by_education,day_std_by_education,day_max_by_education,day_median_by_education,duration_mean_by_education,duration_std_by_education,duration_max_by_education,duration_median_by_education,campaign_mean_by_education,campaign_std_by_education,campaign_max_by_education,campaign_median_by_education,pdays_mean_by_education,pdays_std_by_education,pdays_max_by_education,pdays_median_by_education,previous_mean_by_education,previous_std_by_education,previous_max_by_education,previous_median_by_education,age_mean_by_default,age_std_by_default,age_max_by_default,age_median_by_default,balance_mean_by_default,balance_std_by_default,balance_max_by_default,balance_median_by_default,day_mean_by_default,day_std_by_default,day_max_by_default,day_median_by_default,duration_mean_by_default,duration_std_by_default,duration_max_by_default,duration_median_by_default,campaign_mean_by_default,campaign_std_by_default,campaign_max_by_default,campaign_median_by_default,pdays_mean_by_default,pdays_std_by_default,pdays_max_by_default,pdays_median_by_default,previous_mean_by_default,previous_std_by_default,previous_max_by_default,previous_median_by_default,age_mean_by_housing,age_std_by_housing,age_max_by_housing,age_median_by_housing,balance_mean_by_housing,balance_std_by_housing,balance_max_by_housing,balance_median_by_housing,day_mean_by_housing,day_std_by_housing,day_max_by_housing,day_median_by_housing,duration_mean_by_housing,duration_std_by_housing,duration_max_by_housing,duration_median_by_housing,campaign_mean_by_housing,campaign_std_by_housing,campaign_max_by_housing,campaign_median_by_housing,pdays_mean_by_housing,pdays_std_by_housing,pdays_max_by_housing,pdays_median_by_housing,previous_mean_by_housing,previous_std_by_housing,previous_max_by_housing,previous_median_by_housing,age_mean_by_loan,age_std_by_loan,age_max_by_loan,age_median_by_loan,balance_mean_by_loan,balance_std_by_loan,balance_max_by_loan,balance_median_by_loan,day_mean_by_loan,day_std_by_loan,day_max_by_loan,day_median_by_loan,duration_mean_by_loan,duration_std_by_loan,duration_max_by_loan,duration_median_by_loan,campaign_mean_by_loan,campaign_std_by_loan,campaign_max_by_loan,campaign_median_by_loan,pdays_mean_by_loan,pdays_std_by_loan,pdays_max_by_loan,pdays_median_by_loan,previous_mean_by_loan,previous_std_by_loan,previous_max_by_loan,previous_median_by_loan,age_mean_by_contact,age_std_by_contact,age_max_by_contact,age_median_by_contact,balance_mean_by_contact,balance_std_by_contact,balance_max_by_contact,balance_median_by_contact,day_mean_by_contact,day_std_by_con

In [38]:
for c in df1.columns:
    if df1[c].is_nan().sum() > 0:
        print(c)

balance_std_by_age2
day_std_by_age2
duration_std_by_age2
campaign_std_by_age2
pdays_std_by_age2
previous_std_by_age2
age_std_by_balance2
day_std_by_balance2
duration_std_by_balance2
campaign_std_by_balance2
pdays_std_by_balance2
previous_std_by_balance2
age_std_by_duration2
balance_std_by_duration2
day_std_by_duration2
campaign_std_by_duration2
pdays_std_by_duration2
previous_std_by_duration2
age_std_by_campaign2
balance_std_by_campaign2
day_std_by_campaign2
duration_std_by_campaign2
pdays_std_by_campaign2
previous_std_by_campaign2
age_std_by_pdays2
balance_std_by_pdays2
duration_std_by_pdays2
campaign_std_by_pdays2
previous_std_by_pdays2
age_std_by_previous2
balance_std_by_previous2
day_std_by_previous2
duration_std_by_previous2
campaign_std_by_previous2
pdays_std_by_previous2


In [40]:
df = pl.DataFrame({"x": 0})

In [35]:
def low_std_columns_df(
    df: pl.DataFrame,
    cols: list[str] | None = None,
    threshold: float = 1e-7,
    ddof: int = 0,
) -> list[str]:
    """
    df の cols について標準偏差 <= threshold の列名を返す
    （数値以外は自動で除外してもOK。ここでは明示 cols を優先）
    """
    if cols is None:
        # 数値列だけ自動抽出したい場合はこれでもOK
        num_types = {pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                     pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                     pl.Float32, pl.Float64}
        cols = [c for c,t in df.schema.items() if t in num_types]

    # 安定のため float にキャストしてから std
    std_row = (
        df.select([pl.col(c).cast(pl.Float64).std(ddof=ddof).alias(c) for c in cols])
          .row(0)
    )
    names = cols
    values = std_row  # tuple same order as names
    low_cols = [c for c, s in zip(names, values)
                if (s is None) or (not np.isfinite(s)) or (s < 1e-4)]
    return low_cols

In [36]:
cols = low_std_columns_df(df2)
print(len(cols), cols)

249 ['day_max_by_job', 'pdays_median_by_job', 'previous_median_by_job', 'day_max_by_marital', 'campaign_median_by_marital', 'pdays_median_by_marital', 'previous_median_by_marital', 'age_max_by_education', 'day_max_by_education', 'campaign_median_by_education', 'pdays_median_by_education', 'previous_median_by_education', 'age_median_by_default', 'day_max_by_default', 'campaign_median_by_default', 'pdays_median_by_default', 'previous_median_by_default', 'day_max_by_housing', 'campaign_median_by_housing', 'pdays_max_by_housing', 'pdays_median_by_housing', 'previous_median_by_housing', 'age_median_by_loan', 'day_max_by_loan', 'day_median_by_loan', 'campaign_median_by_loan', 'pdays_median_by_loan', 'previous_median_by_loan', 'day_max_by_contact', 'campaign_median_by_contact', 'pdays_max_by_contact', 'pdays_median_by_contact', 'previous_median_by_contact', 'day_max_by_month', 'pdays_median_by_month', 'previous_median_by_month', 'day_max_by_poutcome', 'balance_std_by_age2', 'day_std_by_age2',

In [30]:
def zero_variance_cols(
    df: pl.DataFrame,
    cols: list[str] | None = None,
    drop_nulls: bool = True,
) -> list[str]:
    """
    非欠損のユニーク数 <= 1 をゼロ分散判定にする安全版
    （= 全部同じか、全部NaNか、NaNと1値のみ）
    """
    if cols is None:
        cols = list(df.columns)

    nunique = df.select([
        (pl.col(c).drop_nulls() if drop_nulls else pl.col(c))
        .n_unique().alias(c) for c in cols
    ])
    vals = nunique.row(0)
    return [c for c, k in zip(nunique.columns, vals) if k <= 2]

In [31]:
cols = zero_variance_cols(df1)
print(len(cols), cols)

7 ['default_te2', 'housing_te2', 'loan_te2', 'default_ce', 'housing_ce', 'loan_ce', 'target']


In [16]:
10 - np.nan

nan